In [ ]:
# Pin to the initial Dia commit whose DiaConfig matches the nari-labs/Dia-1.6B Hub schema.
# The current main branch restructured config fields and breaks loading the Hub model.
!pip install -q git+https://github.com/nari-labs/dia@92206cfb33fbb599166c134d20d247b3c6bd5919 soundfile

In [2]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none — switch Runtime to T4")

GPU: Tesla T4


In [3]:
from dia.model import Dia

print("Loading Dia-1.6B...")
model = Dia.from_pretrained("nari-labs/Dia-1.6B")
print("Model ready.")

Loading Dia-1.6B...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/941 [00:00<?, ?B/s]

RuntimeError: Error loading model from Hugging Face Hub (nari-labs/Dia-1.6B)

In [ ]:
import re

# Paste your script here, or replace with: open("your_script.txt").read()
RAW_SCRIPT = """\
Maya: Kate says she's saving herself—classic.
Ken: Well actually, Lost treats her "escape mindset" like the Island has a billing department.
Maya: You think you're leaving and the Island's like, Cute. Anyway.
Ken: Hey, Maya. Welcome back to Lost Deep Dive. Today we're doing a Kate-focused dive into how Lost uses her motives—escape, control, protection, guilt—to build mythology.
Maya: And I'm already bracing for the part where we're told she's just reacting.
"""

def to_dia_format(text: str) -> str:
    """Convert 'Name: dialogue' lines to Dia's inline [S1]/[S2] format."""
    speaker_map: dict = {}
    segments = []
    for line in text.splitlines():
        m = re.match(r"^([A-Za-z][A-Za-z0-9 _-]*):\s*(.+)$", line.strip())
        if not m:
            continue
        name, dialogue = m.group(1).strip(), m.group(2).strip()
        if name not in speaker_map:
            speaker_map[name] = f"[S{len(speaker_map) + 1}]"
        segments.append(f"{speaker_map[name]} {dialogue}")
    print("Speaker mapping:", speaker_map)
    return " ".join(segments)

script = to_dia_format(RAW_SCRIPT)
print("\nDia input:")
print(script)

In [ ]:
print("Generating audio...")
audio = model.generate(script, verbose=True)
print("Done.")

In [ ]:
import soundfile as sf
from IPython.display import Audio, display

SAMPLE_RATE = 44100
OUT_FILE = "episode_clip_dia.wav"

sf.write(OUT_FILE, audio, SAMPLE_RATE)
print(f"Saved {len(audio)/SAMPLE_RATE:.1f}s → {OUT_FILE}")
display(Audio(audio, rate=SAMPLE_RATE))